In [3]:
import pickle 
import pandas as pd 
with open("statesDict.pkl", "rb") as f:
    sd = pickle.load(f)
pd.Series([b for a,b,c in sd.values()]).unique()

array(['gourmet food', 'sports', 'dressing', 'two-dimensional element',
       'parent-child', 'healthy', 'history', 'take a photo casually',
       'information', 'film and tv series', 'science', 'funny',
       'religion', 'zodiac sign', 'travel', 'beauty makeup', 'animal',
       'life', 'military', 'law', 'educate', 'high-tech digital', 'dance',
       'emotion', 'read', 'talent', 'car', 'photography', 'music', 'game',
       'finance', 'celebrity entertainment', 'appearance',
       'real estate and home furnishings',
       'strange man and strange phenomenon', 'three rural areas'],
      dtype=object)

In [ ]:
from bs4 import BeautifulSoup
from tqdm import tqdm  
from datetime import datetime
from scipy.stats import uniform
from dateutil.parser import parse

import itertools
import requests
import numpy as np 
import pandas as pd 
import os
import pickle
from scipy.interpolate import UnivariateSpline
from tqdm import tqdm

In [141]:
def save_obj(obj, name, folder="cache"):
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, f"{name}.pkl"), "wb") as f:
        pickle.dump(obj, f)

def load_obj(name, folder="cache"):
    path = os.path.join(folder, f"{name}.pkl")
    if os.path.exists(path):
        print(f"{name} cargado desde caché")
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

def saveStatesDict(states, path="./"):
    statesDict = {s:idx for idx,s in enumerate(states)}
    invertedStatesDict = {idx:s for idx,s in enumerate(states)}
    save_obj(obj=statesDict, name="statesDict", folder=path)
    save_obj(obj=invertedStatesDict, name="invertedStatesDict", folder=path)

def getStatesDict(path="./"):
    statesDict = load_obj("statesDict", folder=path)
    invertedStatesDict = load_obj("invertedStatesDict", folder=path)
    return statesDict, invertedStatesDict

def getTransitions(path="./"):
    return np.array(load_obj("matrizDeTransicion", folder=path))

def getCategoriesDict(path="./"):
    return load_obj("categoriesDict", folder=path)

In [149]:
def bulidTransitionsMatrix(dataframeVideos, path="./"):
    states_likes = ["like", "noReaction", "dislike"]
    states_category = dataframeVideos["categoryId"].unique().tolist()
    states_behavior = ["fastSkip", "slowSkip", "outApp"]

    states = list(itertools.product(states_likes, states_category, states_behavior))
    uniform_init = 1 / len(states)
    Transitions = [[uniform_init for j in range(len(states))] for i in range(len(states))]

    statesDict = {s:idx for idx,s in enumerate(states)}
    invertedStatesDict = {idx:s for idx,s in enumerate(states)}
    save_obj(obj=statesDict, name="statesDict", folder=path)
    save_obj(obj=invertedStatesDict, name="invertedStatesDict", folder=path)

    cols = ["categoryId", "id"]
    categoriesDict = dataframeVideos[cols].groupby(by=["categoryId"]).agg(lambda x: x.values).to_dict()["id"]
    save_obj(obj=categoriesDict, name="categoriesDict", folder=path)
    return Transitions

def iterarcion_MC(estado, T):
    num_aleatorio=uniform.rvs(size=1)[0]
    k=0
    sum=T[estado, 0]
    while sum<num_aleatorio:
        k=k+1
        sum=sum+T[estado, k]
    return k

def random_iterarcion_MC(estado, T):
    availableStates = [idx for idx, prob in enumerate(T[estado,:]) if prob != 0]
    if len(availableStates) == 0:
        print(f"Encontro una transicion con p=0, {estado}")
        return estado
    random_state = np.random.choice(availableStates, 1)[0]
    return random_state

def simulacion_MC(estado, tiempo, T):
    cadena = [estado]
    for i in range(tiempo):
        estado=iterarcion_MC(estado, T)
        cadena.append(estado)
    return cadena

In [159]:
def getNextStateRecomendation(currentState, Transitions, exploreOrExploit="explore"):
    if exploreOrExploit == "exploit":
        nextState = iterarcion_MC(currentState, Transitions)
    nextState = random_iterarcion_MC(currentState, Transitions)
    return nextState

def getRecomendation(userState=('like', 'Music', 'fastSkip'), idx=0, exploreOrExploit="explore", pathTransition="./", pathStatesDict="./", pathCategoriesDict="./"): # Explore True, Exploit False
    userState = (userState["receiveLike"], userState["categoryId"], userState["behavior"])

    Transitions = getTransitions(pathTransition)
    categoriesDict = getCategoriesDict(pathCategoriesDict)
    statesDict, invertedStatesDict = getStatesDict(pathStatesDict)
    
    currentState = statesDict[userState]
    # return currentState, Transitions, exploreOrExploit
    idStateRecomendated = getNextStateRecomendation(currentState, Transitions, exploreOrExploit=exploreOrExploit)
    nextState = invertedStatesDict[idStateRecomendated]

    like, category, behavior = nextState
    videosIdRecommended = categoriesDict[category][-1]
    return like, category, behavior, videosIdRecommended

In [4]:
DURATION_SHORTS = 61
DURATION_LONG = 600

def getCategoryLabel(k):
    lbs = k.split(" ")
    if len(lbs) > 1:
        lb = ''.join([l[0] if l[0] != '&' else '' for l in lbs])
    else:
        lb = lbs[0][:2]
    return lb
data_df = pd.read_csv("./youtube.csv",
                      dtype={
                          "id": "string",
                          # "publishedAt": "string",
                          "title": "string",
                          "categoryId": "string",
                          "duration": int,
                          "viewCount": int,
                          "countryCode": "string",
                          # "date": "string",
                          "retentionCurve": "string"},
                      parse_dates=["publishedAt", "date"],
                      date_parser=lambda x: parse(x)
                      )
age_df = pd.read_csv("./youtube_video_age.csv")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23232\2876241371.py:11: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  data_df = pd.read_csv("./youtube.csv",


In [ ]:
# LIMPIEZA PREVIA | YA ME ASEGURE QUE NO HAY OTROS VIDEOS QUE DUREN MAS DE 60 SEC
# USANDO REQUESTS AL MISMO VIDEO EN YOUTUBE
temp = data_df[data_df["duration"] <= 60]
time_videos = [-1, 395*60+31, 715*60+1, -1]
delete_videos = []
for idx, video_idx in enumerate(data_df[data_df["duration"] == 0].index):
    if (time_videos[idx] == -1) | (time_videos[idx] > 60):
        delete_videos.append(video_idx)
        continue
    temp.loc[video_idx, "duration"] = time_videos[idx]
temp = temp.drop(delete_videos)
# temp.head()
del data_df 
del age_df


In [ ]:
# temp.to_csv("videosLessThan60Seconds.csv", index=False)
# path = "videosLessThan60Seconds.csv"
# temp = pd.read_csv(path)


# retention_dict = load_obj(name = "retention_dict", folder="./")
# Transitions = bulidTransitionsMatrix(dataframeVideos = temp)
# save_obj(Transitions, "matrizDeTransicion", folder="./")
# del Transitions

like, category, behavior, videosIdRecommended = getRecomendation(exploreOrExploit="exploit")
print(like, category, behavior, videosIdRecommended)


matrizDeTransicion cargado desde caché
categoriesDict cargado desde caché
statesDict cargado desde caché
invertedStatesDict cargado desde caché
dislike Music outApp 01-HFO4seB4


In [16]:
import pickle
def loadPickle(name):
    with open(name, "rb") as f:
        return pickle.load(f)
s = loadPickle("statesDict.pkl")

In [ ]:
len(s)  

216

In [10]:
# pd.Series([b for a,b,c in s.values()]).unique()
df = pd.read_csv("cats_original.csv")

In [12]:
df[df["retranslate"] == "two-dimensional element"]

,category_level,root_id,parent_id,category_id,retranslate
137,1,20,20,20,two-dimensional element


In [13]:
df = pd.read_csv("interacionesHistorico.csv")

In [15]:
df[df["categoria"] == "two-dimensional element"]

,usuarioId,videoid,interaccionId,hora,fecha,root_id,parent_id,category_id,duration,DaMeGusta,tiempoVisto,categoria
262,3,80358762371,1663478206,13,20220918,20,20,20,17.366,False,2,two-dimensional element
271,3,82483115189,1663673695,19,20220920,20,20,20,86.366,False,89,two-dimensional element
297,3,83184585173,1663473799,12,20220918,20,20,20,17.316,False,34,two-dimensional element
309,3,83394684630,1663476055,12,20220918,20,20,20,15.816,True,18,two-dimensional element
329,3,83616724953,1663760673,20,20220921,20,20,20,94.666,False,97,two-dimensional element
...,...,...,...,...,...,...,...,...,...,...,...,...
1018716,9993,84114974076,1663392045,13,20220917,20,20,20,16.149,False,2,two-dimensional element
1018746,9993,84202603073,1663628768,7,20220920,20,20,20,19.200,False,2,two-dimensional element
1019430,9998,84202603073,1663490552,16,20220918,20,20,20,19.200,False,7,two-dimensional element
1019454,9998,84402817780,1663821643,12,20220922,20,20,20,148.680,True,57,two-dimensional element


In [ ]:
path_lengths = dict(nx.all_pairs_shortest_path_length(G_lcc))

# 3. Obtener una lista ordenada de los nodos
# Esta lista definirá el orden de las filas y columnas en nuestra matriz.
nodes_order = list(G_lcc.nodes())

# 4. Crear un mapeo de cada nodo a su índice en la matriz (0, 1, 2, ...)
node_to_index = {node: i for i, node in enumerate(nodes_order)}

# 5. Inicializar la matriz de costes con ceros
num_nodes = len(nodes_order)
cost_matrix = np.zeros((num_nodes, num_nodes))

# 6. Poblar la matriz de costes con las longitudes de los caminos
for source_node, lengths in path_lengths.items():
    i = node_to_index[source_node]
    for target_node, length in lengths.items():
        j = node_to_index[target_node]
        cost_matrix[i, j] = length